In [ ]:
import re
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# step 1
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=128):
        super().__init__()
        # 1. turns each word's ID number into 64 numbers that describe its meaning
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)   # padding_idx=0: ID 0 is a blank, ignore it
        # 2. reads the review word by word, keeping a memory of what it has read
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        # 3. switches off half the memory at random while training, so it cannot just memorise
        self.drop = nn.Dropout(0.5)
        # 4. turns the final memory into 2 scores: negative, positive
        self.final = nn.Linear(hidden_size, 2)

    def forward(self, text_tensor):
        # text_tensor shape: (batch_size, max_length)
        embedded = self.embedding(text_tensor)      # [batch, 200 IDs] -> [batch, 200, 64]
        _, (hidden, _) = self.lstm(embedded)        # keep only the final memory, ignore the per-word notes
        out = self.drop(hidden[-1])
        return self.final(out)


In [ ]:
#step 2
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"using device-{device}")


dataframe = pd.read_csv('IMDB_Dataset.csv')

# 70% to study, the other 30% for the exam. random_state=42 gives the same split every run
training_df = dataframe.sample(frac=0.7, random_state=42)
test_df = dataframe.drop(training_df.index)


def clean_text(text):
    """Basic text cleaning: lowercase and strip punctuation."""
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.split()


# turn every review into a list of plain words
train_words = training_df["review"].apply(clean_text)
test_words = test_df["review"].apply(clean_text)


# the dictionary: give every common word its own ID number
counts = Counter()
for words in train_words:
    counts.update(words)                 # count how often each word shows up

vocab = {"<PAD>": 0, "<UNK>": 1}         # 0 = blank filler, 1 = a word we have never seen
for word, _ in counts.most_common(10000):
    vocab[word] = len(vocab)             # give this word the next free number


MAX_LEN = 200

def encode(words):
    """Turn a list of words into exactly 200 ID numbers."""
    ids = []
    for word in words[-MAX_LEN:]:        # keep the LAST 200 words (the verdict is usually at the end)
        ids.append(vocab.get(word, 1))   # look the word up, 1 if it is not in the dictionary

    blanks = [0] * (MAX_LEN - len(ids))  # short review? fill the FRONT with blanks
    return blanks + ids


# the reviews become blocks of numbers, the answers become 0 and 1
X_train = torch.tensor([encode(words) for words in train_words])
y_train = torch.tensor(training_df["sentiment"].map({"negative": 0, "positive": 1}).values)
X_test = torch.tensor([encode(words) for words in test_words])
y_test = torch.tensor(test_df["sentiment"].map({"negative": 0, "positive": 1}).values)

# hand them out in batches. shuffle the study reviews, no need to shuffle the exam
train_dataloader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_dataloader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

print(f"{len(X_train)} training reviews, {len(X_test)} test reviews, {len(vocab)} words in the dictionary")


In [ ]:
#step 3

epochs = 10
model = SentimentModel(vocab_size=len(vocab)).to(device)

loss_fun = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


def check_score():
    """Grade the model on the test reviews, give back the % it got right."""
    model.eval()                                  # exam mode: dropout switched off
    correct = 0
    total = 0
    with torch.no_grad():                         # just guessing, no learning
        for reviews, labels in test_dataloader:
            reviews, labels = reviews.to(device), labels.to(device)
            guess = model(reviews)
            answer = guess.argmax(dim=1)          # whichever of the 2 scores is higher
            correct += (answer == labels).sum().item()
            total += len(labels)
    model.train()                                 # back to learning mode
    return correct / total * 100


best_score = 0

model.train()
for epoch in range(epochs):
    total_loss = 0

    for reviews, labels in train_dataloader:
        reviews, labels = reviews.to(device), labels.to(device)

        optimizer.zero_grad()               # wipe the last batch's corrections
        guess = model(reviews)              # what does it think these are?
        loss = loss_fun(guess, labels)      # how wrong was it?
        loss.backward()                     # work out which way to nudge things
        optimizer.step()                    # nudge

        total_loss += loss.item()

    # grade after every epoch, so you can see when it stops improving
    score = check_score()
    print(f"Epoch {epoch+1}: avg loss {total_loss / len(train_dataloader):.4f}  test {score:.1f}%")

    # keep a copy of the best model so far, so a bad later epoch cannot spoil it
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), "sentiment_model.pth")

print(f"best test score- {best_score:.1f}% (saved to sentiment_model.pth)")


In [7]:
#step 4

# bring back the best epoch's model (the one saved during training) and grade it
model.load_state_dict(torch.load("sentiment_model.pth", map_location=device))

print(f"final test score- {check_score():.1f}%")


final test score- 87.9%


In [ ]:
#step 5 - try it on your own reviews

def how_does_it_feel(review):
    """Say whether one review is positive or negative, and how sure the model is."""
    words = clean_text(review)                        # same cleaning as the training reviews
    numbers = torch.tensor([encode(words)]).to(device)  # one review, wrapped in a batch of one

    model.eval()
    with torch.no_grad():
        scores = model(numbers)                       # 2 raw scores: negative, positive
        chances = torch.softmax(scores, dim=1)[0]     # turn them into percentages adding up to 100

    if chances[1] > chances[0]:
        return "positive", chances[1].item() * 100
    return "negative", chances[0].item() * 100


# change these or add your own
for review in [
    "This movie was wonderful. Brilliant acting and I loved every minute of it.",
    "What a waste of two hours. Boring plot, terrible acting, I want my money back.",
    "I expected it to be awful, but honestly it turned out to be great.",
    "It was not good.",
]:
    mood, sureness = how_does_it_feel(review)
    print(f"{mood} ({sureness:.1f}% sure)  <-  {review}")
